# GPT-2 From Scratch

### Overview

This project implements a GPT-2 style decoder-only Transformer language model completely from scratch using PyTorch. The implementation is inspired by Andrej Karpathy's educational series while following a clean, modular software engineering approach.

### Objectives

- Build a GPT-2 style language model from scratch
- Understand every component of the Transformer architecture
- Train the model on a text dataset
- Generate coherent text
- Develop a Streamlit interface for inference

### Technologies Used

- Python
- PyTorch
- NumPy
- Matplotlib
- Streamlit
- Google Colab / VS Code

In [1]:
import os
import math
import random

import torch
import torch.nn as nn
from torch.nn import functional as F

import matplotlib.pyplot as plt

torch.manual_seed(42)

print("PyTorch Version:", torch.__version__)
print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")

PyTorch Version: 2.7.1+cpu
Device: CPU


In [2]:
# Hyperparameters

batch_size = 64
block_size = 256
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = "cuda" if torch.cuda.is_available() else "cpu"
eval_iters = 200

n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2

# 1. Dataset Loading

In this section, we load the Tiny Shakespeare dataset, which will be used to train our GPT-2 model. This dataset consists of dialogues from Shakespeare's plays and serves as the training corpus for our language model.

In [3]:
DATA_PATH = "../data/input.txt"

In [4]:
# Load Dataset

with open(DATA_PATH, "r", encoding="utf-8") as file:
    text = file.read()

print("Dataset Loaded Successfully!")

Dataset Loaded Successfully!


In [5]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



# 2. Dataset Statistics

Before building the tokenizer, let's understand the dataset by checking its size and the number of unique characters present.

In [6]:
print(f"Dataset Length : {len(text):,} characters")
print(f"Unique Characters : {len(set(text))}")

Dataset Length : 1,115,394 characters
Unique Characters : 65


In [7]:
chars = sorted(list(set(text)))
print(chars)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [8]:
print("Vocabulary Size:", len(chars))

Vocabulary Size: 65


# 3. Character Vocabulary

GPT does not understand raw text. The first step is to convert every unique character into a unique integer. This mapping forms the vocabulary used by the tokenizer.

In [9]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("Vocabulary Size:", vocab_size)

Vocabulary Size: 65


In [10]:
# Character to Integer Mapping

stoi = {ch: i for i, ch in enumerate(chars)}

# Integer to Character Mapping

itos = {i: ch for i, ch in enumerate(chars)}

In [11]:
print(stoi)

{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}


In [12]:
print(itos)

{0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R', 31: 'S', 32: 'T', 33: 'U', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Z', 39: 'a', 40: 'b', 41: 'c', 42: 'd', 43: 'e', 44: 'f', 45: 'g', 46: 'h', 47: 'i', 48: 'j', 49: 'k', 50: 'l', 51: 'm', 52: 'n', 53: 'o', 54: 'p', 55: 'q', 56: 'r', 57: 's', 58: 't', 59: 'u', 60: 'v', 61: 'w', 62: 'x', 63: 'y', 64: 'z'}


# 4. Character Tokenizer

A tokenizer converts text into numerical representations that a neural network can process. In this implementation, we use a character-level tokenizer where every unique character is assigned a unique integer ID.

In [13]:
# Encode Function

encode = lambda s: [stoi[c] for c in s]

In [14]:
# Decode Function

decode = lambda l: ''.join([itos[i] for i in l])

In [15]:
sample_text = "Hello GPT"

encoded = encode(sample_text)

print("Original Text:")
print(sample_text)

print("\nEncoded:")
print(encoded)

Original Text:
Hello GPT

Encoded:
[20, 43, 50, 50, 53, 1, 19, 28, 32]


In [16]:
decoded = decode(encoded)

print("Decoded Text:")
print(decoded)

Decoded Text:
Hello GPT


In [17]:
print(sample_text == decoded)

True


In [18]:
sentence = "Machine Learning"

encoded_sentence = encode(sentence)
decoded_sentence = decode(encoded_sentence)

print(encoded_sentence)
print(decoded_sentence)

[25, 39, 41, 46, 47, 52, 43, 1, 24, 43, 39, 56, 52, 47, 52, 45]
Machine Learning


# 5. Dataset Encoding

The tokenizer converts individual strings into integer sequences. In this step, we encode the entire dataset and convert it into a PyTorch tensor so it can be used for model training.

In [19]:
# Encoding Entire Dataset

data = torch.tensor(encode(text), dtype=torch.long)

print(data)

tensor([18, 47, 56,  ..., 45,  8,  0])


In [20]:
print(data.shape)

torch.Size([1115394])


In [21]:
print(data.dtype)

torch.int64


# 6. Train and Validation Split

To evaluate model performance, we split the dataset into training and validation sets. The model learns from the training data, while the validation data is used to measure generalization.

In [22]:
# 90% Training
# 10% Validation

n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

In [23]:
print("Training Data:", train_data.shape)
print("Validation Data:", val_data.shape)

Training Data: torch.Size([1003854])
Validation Data: torch.Size([111540])


# 7. Context Window (Block Size)

Instead of training on the entire dataset at once, GPT learns from small continuous chunks of text called context windows. The length of this context window is defined by the block size.

In [24]:
print(train_data[:block_size + 1])

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
        47, 59, 57,  1, 47, 57,  1, 41, 

In [25]:
x = train_data[:block_size]
y = train_data[1:block_size + 1]

print("Input Shape :", x.shape)
print("Target Shape:", y.shape)

Input Shape : torch.Size([256])
Target Shape: torch.Size([256])


In [26]:
for t in range(block_size):

    context = x[:t + 1]
    target = y[t]

    print(f"Context: {context.tolist()} --> Target: {target}")

Context: [18] --> Target: 47
Context: [18, 47] --> Target: 56
Context: [18, 47, 56] --> Target: 57
Context: [18, 47, 56, 57] --> Target: 58
Context: [18, 47, 56, 57, 58] --> Target: 1
Context: [18, 47, 56, 57, 58, 1] --> Target: 15
Context: [18, 47, 56, 57, 58, 1, 15] --> Target: 47
Context: [18, 47, 56, 57, 58, 1, 15, 47] --> Target: 58
Context: [18, 47, 56, 57, 58, 1, 15, 47, 58] --> Target: 47
Context: [18, 47, 56, 57, 58, 1, 15, 47, 58, 47] --> Target: 64
Context: [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64] --> Target: 43
Context: [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43] --> Target: 52
Context: [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52] --> Target: 10
Context: [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10] --> Target: 0
Context: [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0] --> Target: 14
Context: [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0, 14] --> Target: 43
Context: [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0, 14,

## Context and Target Visualization

The model learns by observing a sequence of characters (context) and predicting the next character (target). Along with token IDs, we also display the decoded text to better understand the learning process.

In [27]:
for t in range(10):   # Display only first 10 examples

    context = x[:t + 1]
    target = y[t]

    print("=" * 60)
    print(f"Context IDs   : {context.tolist()}")
    print(f"Context Text  : '{decode(context.tolist())}'")
    print(f"Target ID     : {target.item()}")
    print(f"Target Char   : '{decode([target.item()])}'")

Context IDs   : [18]
Context Text  : 'F'
Target ID     : 47
Target Char   : 'i'
Context IDs   : [18, 47]
Context Text  : 'Fi'
Target ID     : 56
Target Char   : 'r'
Context IDs   : [18, 47, 56]
Context Text  : 'Fir'
Target ID     : 57
Target Char   : 's'
Context IDs   : [18, 47, 56, 57]
Context Text  : 'Firs'
Target ID     : 58
Target Char   : 't'
Context IDs   : [18, 47, 56, 57, 58]
Context Text  : 'First'
Target ID     : 1
Target Char   : ' '
Context IDs   : [18, 47, 56, 57, 58, 1]
Context Text  : 'First '
Target ID     : 15
Target Char   : 'C'
Context IDs   : [18, 47, 56, 57, 58, 1, 15]
Context Text  : 'First C'
Target ID     : 47
Target Char   : 'i'
Context IDs   : [18, 47, 56, 57, 58, 1, 15, 47]
Context Text  : 'First Ci'
Target ID     : 58
Target Char   : 't'
Context IDs   : [18, 47, 56, 57, 58, 1, 15, 47, 58]
Context Text  : 'First Cit'
Target ID     : 47
Target Char   : 'i'
Context IDs   : [18, 47, 56, 57, 58, 1, 15, 47, 58, 47]
Context Text  : 'First Citi'
Target ID     : 64
T

# 8. Batch Generation

Instead of training on the entire dataset at once, we randomly sample multiple small sequences (batches). This improves training efficiency and helps the model learn different parts of the dataset.

In [28]:
# Batch Generation

def get_batch(split):
    data_source = train_data if split == "train" else val_data

    # Random starting positions
    ix = torch.randint(len(data_source) - block_size, (batch_size,))

    # Input sequences
    x = torch.stack([data_source[i:i + block_size] for i in ix])

    # Target sequences
    y = torch.stack([data_source[i + 1:i + block_size + 1] for i in ix])

    x = x.to(device)
    y = y.to(device)

    return x, y

In [29]:
xb, yb = get_batch("train")

print("Input Shape :", xb.shape)
print("Target Shape:", yb.shape)

Input Shape : torch.Size([64, 256])
Target Shape: torch.Size([64, 256])


In [30]:
print("First Input Sequence:\n")
print(decode(xb[0].tolist()))

print("\n" + "="*70 + "\n")

print("First Target Sequence:\n")
print(decode(yb[0].tolist()))

First Input Sequence:

pey the
Great. Pompey, you are partly a bawd, Pompey,
howsoever you colour it in being a tapster, are you
not? come, tell me true: it shall be the better for you.

POMPEY:
Truly, sir, I am a poor fellow that would live.

ESCALUS:
How would you live, Pompey


First Target Sequence:

ey the
Great. Pompey, you are partly a bawd, Pompey,
howsoever you colour it in being a tapster, are you
not? come, tell me true: it shall be the better for you.

POMPEY:
Truly, sir, I am a poor fellow that would live.

ESCALUS:
How would you live, Pompey?


In [31]:
print("Input IDs:")
print(xb[0][:20])

print("\nTarget IDs:")
print(yb[0][:20])

Input IDs:
tensor([54, 43, 63,  1, 58, 46, 43,  0, 19, 56, 43, 39, 58,  8,  1, 28, 53, 51,
        54, 43])

Target IDs:
tensor([43, 63,  1, 58, 46, 43,  0, 19, 56, 43, 39, 58,  8,  1, 28, 53, 51, 54,
        43, 63])
